## BLOCK 1 — Install UVLM + Load Model
Installs UVLM from GitHub, mounts Google Drive, and presents a widget UI to load a vision-language model.

In [ ]:
# ===============================================
# BLOCK 1 — Install UVLM + Load Model
# ===============================================

# Install UVLM from GitHub
!pip install -q git+https://github.com/perezjoan/UVLM.git

from uvlm import load_model
from uvlm.registry import MODEL_CHOICES
from uvlm.utils import get_hf_token

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import ipywidgets as widgets
from IPython.display import display

use_token_widget = widgets.Checkbox(value=True, description="Use Hugging Face token (from Colab secrets)")
model_widget = widgets.Dropdown(options=list(MODEL_CHOICES.keys()), value="[Qwen]  Qwen2.5-VL 7B Instruct", description="Model")
precision_widget = widgets.Dropdown(
    options=[("4-bit (bnb, low VRAM)", "4bit"), ("8-bit (bnb)", "8bit"), ("FP16 / auto", "fp16")],
    value="4bit", description="Precision")
low_cpu_widget = widgets.Checkbox(value=True, description="low_cpu_mem_usage")
device_map_widget = widgets.Dropdown(
    options=[("GPU only (cuda:0)", "cuda0"), ("Auto (accelerate decides)", "auto"),
             ("GPU + CPU offload", "offload")],
    value="auto", description="Placement")
load_button = widgets.Button(description="Load model", button_style="success")
status_html = widgets.HTML(value="")

def on_load(b):
    global model_ctx
    status_html.value = "<b>Loading model...</b>"
    try:
        hf_token = get_hf_token() if use_token_widget.value else None
        model_ctx = load_model(
            model_name=model_widget.value,
            precision=precision_widget.value,
            device_map=device_map_widget.value,
            low_cpu_mem_usage=low_cpu_widget.value,
            hf_token=hf_token,
        )
        status_html.value = (
            f"<b>Model loaded</b><br>"
            f"Time: <b>{model_ctx['load_time_minutes']:.2f}</b> min<br>"
            f"{model_widget.value} on {model_ctx['gpu_name']} ({precision_widget.value})"
        )
    except Exception as e:
        status_html.value = f"<b>Load failed</b><br><code>{e}</code>"
        raise

load_button.on_click(on_load)
display(widgets.VBox([use_token_widget, model_widget, precision_widget,
                      low_cpu_widget, device_map_widget, load_button, status_html]))


## BLOCK 2 — Define Tasks + Settings
Set the Google Drive image folder, define tasks, and configure generation parameters. Click **Apply** when done.

In [ ]:
# ===============================================
# BLOCK 2 — Define Tasks + Settings
# ===============================================

folder_widget = widgets.Text(
    description="Image folder",
    placeholder="Folder name in MyDrive/",
    layout=widgets.Layout(width="60%"),
)

from uvlm.prompts import TASK_TYPES, ADVANCED_REASONING_FORMATS, ADVANCED_REASONING_MAX_TOKENS, build_prompt
from transformers import AutoProcessor
import os
import ipywidgets as widgets
from IPython.display import display

role_box = widgets.Textarea(
    description="Role (global)",
    placeholder="This part will be prepended to ALL tasks",
    layout=widgets.Layout(width="100%", height="90px"),
)

n_tasks_widget = widgets.IntSlider(value=1, min=1, max=10, step=1, description="Nb tasks")
tasks_container = widgets.VBox()

do_sample_widget = widgets.Checkbox(value=True, description="Enable sampling")
temperature_widget = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description="Temperature")
top_p_widget = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.05, description="Top-p")
max_tokens_widget = widgets.IntSlider(value=50, min=1, max=1500, step=1, description="Max tokens")
display_images_widget = widgets.Checkbox(value=True, description="Display images during scoring")
fixed_seed_widget = widgets.Checkbox(value=False, description="Use fixed seed")
seed_value_widget = widgets.IntText(value=42, description="Seed value", disabled=True)

def on_fixed_seed_change(change):
    if change["name"] == "value":
        seed_value_widget.disabled = not change["new"]
fixed_seed_widget.observe(on_fixed_seed_change)

DEFAULT_MIN_PIXELS = 256 * 28 * 28
DEFAULT_MAX_PIXELS = 640 * 28 * 28
qwen_settings_label = widgets.HTML(value="<b>Qwen-specific settings</b> <small>(affects VRAM usage vs image quality)</small>")
min_pixels_widget = widgets.IntText(value=DEFAULT_MIN_PIXELS, description="min_pixels", layout=widgets.Layout(width="50%"))
max_pixels_widget = widgets.IntText(value=DEFAULT_MAX_PIXELS, description="max_pixels", layout=widgets.Layout(width="50%"))
qwen_pixels_help = widgets.HTML(
    value="<small><i>Lower values = less VRAM. Formula: N x 28 x 28. Common: 256x28x28=200704 (min), 640x28x28=501760 (default), 1280x28x28=1003520 (high)</i></small>"
)
qwen_settings_box = widgets.VBox(
    [qwen_settings_label, min_pixels_widget, max_pixels_widget, qwen_pixels_help],
    layout=widgets.Layout(border="1px solid #4a9eff", padding="10px", margin="10px 0px", display="none"),
)

apply_button = widgets.Button(description="Apply paths + tasks + settings", button_style="success")
status_out = widgets.Output()

TASK_WIDGETS = []

def build_task_widgets(n):
    global TASK_WIDGETS
    TASK_WIDGETS = []
    blocks = []
    for i in range(1, n + 1):
        task_type = widgets.Dropdown(
            options=[("Numeric (counts, scores)", "numeric"), ("Category (classifications)", "category"),
                     ("Boolean (yes/no)", "boolean"), ("Text (free-form)", "text")],
            value="numeric", description=f"Type {i}", layout=widgets.Layout(width="60%"))
        col_title = widgets.Text(description=f"Col {i}", value=f"task_{i}", layout=widgets.Layout(width="60%"))
        theory = widgets.Textarea(description=f"Theory {i}", placeholder=f"Scoring rules (task {i})",
                                  layout=widgets.Layout(width="100%", height="110px"))
        task = widgets.Textarea(description=f"Task {i}", placeholder=f"What the model must do (task {i})",
                                layout=widgets.Layout(width="100%", height="80px"))
        fmt = widgets.Textarea(description=f"Format {i}", placeholder=f"Expected output format (task {i})",
                               layout=widgets.Layout(width="100%", height="60px"))
        help_text = widgets.HTML(value="<small><i>Numeric: extracts numbers | Category: returns classification text | Boolean: normalizes to 1/0 | Text: raw response</i></small>")

        advanced_reasoning_checkbox = widgets.Checkbox(value=False, description="Enable advanced reasoning (chain-of-thought)", layout=widgets.Layout(width="auto"))
        adv_help = widgets.HTML(
            value=f"<small><i>Model describes observations + reasoning before answering. Max tokens set to {ADVANCED_REASONING_MAX_TOKENS}. Adds reasoning and truncation columns.</i></small>",
            layout=widgets.Layout(display="none"))
        adv_preview = widgets.HTML(value="", layout=widgets.Layout(display="none"))

        consensus_checkbox = widgets.Checkbox(value=False, description="Enable consensus validation", layout=widgets.Layout(width="auto"))
        consensus_runs = widgets.Dropdown(options=[("2 runs", 2), ("3 runs", 3), ("4 runs", 4), ("5 runs", 5)],
                                          value=2, description="Runs", disabled=True, layout=widgets.Layout(width="30%"))
        numeric_tolerance = widgets.FloatText(value=0.0, description="Tolerance %", layout=widgets.Layout(width="30%"))
        consensus_help = widgets.HTML(
            value="<small><i>Consensus multiplies API calls! Tolerance: for numeric, values within X% are considered equal (0 = exact match)</i></small>",
            layout=widgets.Layout(display="none"))
        consensus_cost = widgets.HTML(value="", layout=widgets.Layout(display="none"))
        consensus_opts = widgets.HBox([consensus_runs, numeric_tolerance], layout=widgets.Layout(display="none"))

        def make_adv_obs(chk, hlp, prv, fmt_w, tt_w):
            def obs(change):
                if change["name"] == "value":
                    enabled = change["new"]
                    hlp.layout.display = "block" if enabled else "none"
                    prv.layout.display = "block" if enabled else "none"
                    fmt_w.disabled = enabled
                    if enabled and tt_w.value != "text":
                        ft = ADVANCED_REASONING_FORMATS.get(tt_w.value, "")
                        prv.value = f"<small><b>Format (auto):</b><br><code style='white-space: pre-wrap;'>{ft}</code></small>"
                        fmt_w.value = ""
                    else:
                        prv.value = ""
            return obs

        def make_type_obs(tol_w, adv_chk, adv_hlp, adv_prv, fmt_w):
            def obs(change):
                if change["name"] == "value":
                    nt = change["new"]
                    tol_w.disabled = nt != "numeric"
                    if nt == "text":
                        adv_chk.value = False
                        adv_chk.disabled = True
                        adv_hlp.layout.display = "none"
                        adv_prv.layout.display = "none"
                        fmt_w.disabled = False
                    else:
                        adv_chk.disabled = False
                        if adv_chk.value:
                            ft = ADVANCED_REASONING_FORMATS.get(nt, "")
                            adv_prv.value = f"<small><b>Format (auto):</b><br><code style='white-space: pre-wrap;'>{ft}</code></small>"
            return obs

        def make_cons_obs(chk, opts, hlp, cost, runs_w, tt_w):
            def obs(change):
                if change["name"] == "value":
                    enabled = change["new"]
                    opts.layout.display = "flex" if enabled else "none"
                    hlp.layout.display = "block" if enabled else "none"
                    cost.layout.display = "block" if enabled else "none"
                    runs_w.disabled = not enabled
                    cost.value = (f"<small style='color: #ff6600;'><b>Cost: {runs_w.value}x API calls per image for this task</b></small>"
                                  if enabled else "")
            return obs

        def make_runs_obs(cost, chk):
            def obs(change):
                if change["name"] == "value" and chk.value:
                    cost.value = f"<small style='color: #ff6600;'><b>Cost: {change['new']}x API calls per image for this task</b></small>"
            return obs

        advanced_reasoning_checkbox.observe(make_adv_obs(advanced_reasoning_checkbox, adv_help, adv_preview, fmt, task_type))
        task_type.observe(make_type_obs(numeric_tolerance, advanced_reasoning_checkbox, adv_help, adv_preview, fmt))
        consensus_checkbox.observe(make_cons_obs(consensus_checkbox, consensus_opts, consensus_help, consensus_cost, consensus_runs, task_type))
        consensus_runs.observe(make_runs_obs(consensus_cost, consensus_checkbox))

        TASK_WIDGETS.append({"task_type": task_type, "col_title": col_title, "theory": theory,
                             "task": task, "format": fmt, "advanced_reasoning": advanced_reasoning_checkbox,
                             "consensus_enabled": consensus_checkbox, "consensus_runs": consensus_runs,
                             "numeric_tolerance": numeric_tolerance})

        block = widgets.VBox(
            [widgets.HTML(f"<b>Task #{i}</b>"), task_type, col_title, task, theory, fmt, help_text,
             widgets.VBox([widgets.HTML("<hr style='margin: 5px 0;'>"),
                           widgets.HTML("<b style='font-size: 0.9em;'>Advanced Reasoning</b>"),
                           advanced_reasoning_checkbox, adv_help, adv_preview],
                          layout=widgets.Layout(margin="5px 0 0 0")),
             widgets.VBox([widgets.HTML("<hr style='margin: 5px 0;'>"),
                           widgets.HTML("<b style='font-size: 0.9em;'>Consensus Validation</b>"),
                           consensus_checkbox, consensus_opts, consensus_help, consensus_cost],
                          layout=widgets.Layout(margin="5px 0 0 0"))],
            layout=widgets.Layout(border="1px solid #ddd", padding="10px", margin="6px 0px 10px 0px"))
        blocks.append(block)

    tasks_container.children = tuple(blocks)

build_task_widgets(n_tasks_widget.value)

def on_n_tasks_change(change):
    if change["name"] == "value":
        build_task_widgets(int(change["new"]))
n_tasks_widget.observe(on_n_tasks_change)

def _get_model_ctx():
    if "model_ctx" not in globals():
        raise RuntimeError("Load a model first (Block 1)")
    return model_ctx

def apply_settings(b):
    global image_folder, output_path, do_sample, temperature, top_p, max_new_tokens
    global display_images, generation_seed, task_specs
    try:
        ctx = _get_model_ctx()
    except RuntimeError as e:
        with status_out:
            status_out.clear_output()
            print(str(e))
        return

    root_path = "/content/drive/MyDrive"
    folder_name = folder_widget.value.strip()
    image_folder = os.path.join(root_path, folder_name)
    backend = ctx["backend"]
    out_name = "Score_Analysis_QwenVL.csv" if backend == "qwen" else "Score_Analysis_LLaVA.csv"
    output_path = os.path.join(image_folder, out_name)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    do_sample = do_sample_widget.value
    temperature = float(temperature_widget.value)
    top_p = float(top_p_widget.value)
    max_new_tokens = int(max_tokens_widget.value)
    display_images = bool(display_images_widget.value)
    generation_seed = int(seed_value_widget.value) if fixed_seed_widget.value else None

    if backend == "qwen":
        new_min = int(min_pixels_widget.value)
        new_max = int(max_pixels_widget.value)
        if new_min != ctx["qwen_min_pixels"] or new_max != ctx["qwen_max_pixels"]:
            print(f"Reloading Qwen processor with new pixel settings: min={new_min}, max={new_max}")
            auth_kwargs = {"token": ctx["hf_token"]} if ctx.get("hf_token") else {}
            ctx["processor"] = AutoProcessor.from_pretrained(
                ctx["model_id"], min_pixels=new_min, max_pixels=new_max, **auth_kwargs)
            ctx["qwen_min_pixels"] = new_min
            ctx["qwen_max_pixels"] = new_max
            print("Processor reloaded.")

    role_txt = role_box.value.strip()
    task_specs = []
    seen_cols = set()
    for i, tw in enumerate(TASK_WIDGETS, start=1):
        task_type = tw["task_type"].value
        col = tw["col_title"].value.strip() or f"task_{i}"
        th = tw["theory"].value.strip()
        tk = tw["task"].value.strip()
        fm = tw["format"].value.strip()
        advanced_reasoning = tw["advanced_reasoning"].value
        if task_type == "text" and advanced_reasoning:
            advanced_reasoning = False
        if advanced_reasoning:
            fm = ADVANCED_REASONING_FORMATS.get(task_type, fm)
        consensus_enabled = tw["consensus_enabled"].value
        c_runs = tw["consensus_runs"].value if consensus_enabled else 1
        n_tol = tw["numeric_tolerance"].value if task_type == "numeric" else 0.0
        if task_type == "text" and consensus_enabled:
            consensus_enabled = False
            c_runs = 1
        if col in seen_cols:
            col = f"{col}_{i}"
        seen_cols.add(col)
        full_prompt = build_prompt(role_txt, tk, th, fm)
        task_specs.append({"column": col, "prompt": full_prompt, "task_type": task_type,
                           "advanced_reasoning": advanced_reasoning, "consensus_enabled": consensus_enabled,
                           "consensus_runs": c_runs, "numeric_tolerance": n_tol / 100.0})

    status_out.clear_output(wait=True)
    with status_out:
        print("Settings applied")
        print(f"Backend: {backend} | Folder: {image_folder}")
        print(f"Output: {output_path}")
        print(f"Tasks: {len(task_specs)} | do_sample={do_sample} temp={temperature} top_p={top_p} max_tokens={max_new_tokens}")
        print(f"Seed: {generation_seed} | Display images: {display_images}")
        for j, spec in enumerate(task_specs, 1):
            flags = []
            if spec["advanced_reasoning"]: flags.append("reasoning")
            if spec["consensus_enabled"]: flags.append(f"consensus={spec['consensus_runs']}x")
            print(f"  {j}. {spec['column']} [{spec['task_type']}]" + (" | " + " | ".join(flags) if flags else ""))

apply_button.on_click(apply_settings)

display(folder_widget, role_box, n_tasks_widget, tasks_container, qwen_settings_box,
        display_images_widget, do_sample_widget, temperature_widget, top_p_widget, max_tokens_widget,
        fixed_seed_widget, seed_value_widget, apply_button, status_out)

def update_qwen_settings_visibility():
    if "model_ctx" in globals() and model_ctx.get("backend") == "qwen":
        qwen_settings_box.layout.display = "block"
        min_pixels_widget.value = model_ctx.get("qwen_min_pixels", DEFAULT_MIN_PIXELS)
        max_pixels_widget.value = model_ctx.get("qwen_max_pixels", DEFAULT_MAX_PIXELS)
    else:
        qwen_settings_box.layout.display = "none"

update_qwen_settings_visibility()


## BLOCK 3 — Run Analysis
Processes all images. Resume mode: already-done images are skipped automatically.

In [ ]:
# ===============================================
# BLOCK 3 — Run Analysis
# ===============================================

from uvlm.batch import run_batch

df = run_batch(
    model_ctx=model_ctx,
    task_specs=task_specs,
    image_folder=image_folder,
    output_path=output_path,
    max_new_tokens=max_new_tokens,
    do_sample=do_sample,
    temperature=temperature,
    top_p=top_p,
    seed=generation_seed,
    display_images=display_images,
)
